In [ ]:
!pip install anthropic pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.8/999.8 kB 15.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
print("Cliente listo ✅")

Cliente listo ✅


In [ ]:
# ====== CELDA 3: construcción de viñetas y prompt ======

# Nombres por situación y origen (del libro de Gilgen)
NOMBRES_TRABAJO = {
    "local":  {"f": "Laura",  "m": "Lukas"},
    "arabe":  {"f": "Salma",  "m": "Ahmad"},
    "eslavo": {"f": "Milena", "m": "Marko"},
}
NOMBRES_FAM_F = ["Anna", "Sylvia"]
NOMBRES_FAM_M = ["David", "Lorenz"]

# Niveles literales (redacción exacta del libro)
SALUD    = {1: "is in good health",    2: "is in poor health"}
FINANZAS = {1: "is in an easy financial situation", 2: "is in a tight financial situation"}
DEDICACION = {1: "is not very dedicated to the job",
              2: "is more or less dedicated to the job",
              3: "is very dedicated to the job"}
DESEMPENO  = {1: "shows below average performance",
              2: "shows average performance",
              3: "shows above average performance"}
ANOS_EMPRESA = {2: "has been with the company for 2 years",
                8: "has been with the company for 8 years",
                16: "has been with the company for 16 years"}
ATENCION = {1: "was not very attentive and helpful in the past",
            2: "was mostly attentive and helpful in the past",
            3: "was very attentive and helpful in the past"}

def pron(g): return "He" if g == "m" else "She"

def _vida(pareja, hijos):
    if pareja == 2 and hijos == 2: return "is a single parent of two children"
    elif pareja == 2 and hijos == 1: return "is single with no children"
    elif pareja == 1 and hijos == 2: return "lives with a partner and has two children"
    else: return "lives with a partner and has no children"

def construir_vineta_trabajo(g, origen, pareja, hijos, salud, dedic, desemp, anos):
    nombre = NOMBRES_TRABAJO[origen][g]; p = pron(g)
    return ". ".join([
        f"{nombre} {_vida(pareja, hijos)}",
        f"{p} {SALUD[salud]}",
        f"{p} {DEDICACION[dedic]}",
        f"{p} {DESEMPENO[desemp]}",
        f"{p} {ANOS_EMPRESA[anos]}",
    ]) + "."

def construir_vineta_familia(g, nombre, pareja, hijos, salud, finanzas, atencion):
    p = pron(g)
    return ". ".join([
        f"{nombre} {_vida(pareja, hijos)}",
        f"{p} {SALUD[salud]}",
        f"{p} {FINANZAS[finanzas]}",
        f"{p} {ATENCION[atencion]}",
    ]) + "."

INTRO_TRABAJO = ("Imagine you are the boss of the three employees described below. "
                 "You can decide how to distribute a sum of CHF {suma} among them as "
                 "end of year bonus money. Please distribute the money among the three "
                 "in the way you consider fair.")
INTRO_FAMILIA = ("Imagine you are distributing an inheritance of CHF {suma} among your "
                 "three grown-up children, described below. Please distribute the money "
                 "in the way you consider fair.")

def prompt_completo(situacion, suma, vinetas):
    suma_fmt = f"{suma:,}"
    if situacion == "trabajo":
        intro = INTRO_TRABAJO.format(suma=suma_fmt); etiqueta = "Employee"
    else:
        intro = INTRO_FAMILIA.format(suma=suma_fmt); etiqueta = "Child"
    cuerpo = "\n".join(f"{etiqueta} {i}: {v}" for i, v in enumerate(vinetas, 1))
    cierre = (f"Even if this involves a decision you do not personally make, give your "
              f"best allocation. The three amounts must add up to CHF {suma_fmt}. "
              f"Respond with ONLY the three amounts in this exact format:\n"
              f"AMOUNTS: <1>, <2>, <3>")
    return f"{intro}\n\n{cuerpo}\n\n{cierre}"

print("Celda 3 lista ✅ — viñetas y prompt definidos")

Celda 3 lista ✅ — viñetas y prompt definidos


In [ ]:
# ====== CELDA 4: generación de sets balanceados ======
import random
from collections import Counter

SEMILLA = 42

NIV_TRABAJO = {
    "genero": ["m", "f"], "origen": ["local", "arabe", "eslavo"],
    "pareja": [1, 2], "hijos": [1, 2], "salud": [1, 2],
    "dedic": [1, 2, 3], "desemp": [1, 2, 3], "anos": [2, 8, 16],
}
NIV_FAMILIA = {
    "genero": ["m", "f"], "pareja": [1, 2], "hijos": [1, 2],
    "salud": [1, 2], "finanzas": [1, 2], "atencion": [1, 2, 3],
}

def _columnas_balanceadas(niveles, n, rng):
    cols = {}
    for dim, vals in niveles.items():
        base = (vals * (n // len(vals) + 1))[:n]
        rng.shuffle(base)
        cols[dim] = base
    return [{dim: cols[dim][i] for dim in niveles} for i in range(n)]

def _nombres_unicos_trabajo(perfiles):
    usados = set()
    for p in perfiles:
        nombre = NOMBRES_TRABAJO[p["origen"]][p["genero"]]
        if nombre in usados:
            for alt in ["local", "arabe", "eslavo"]:
                cand = NOMBRES_TRABAJO[alt][p["genero"]]
                if cand not in usados:
                    p["origen"] = alt; nombre = cand; break
        usados.add(nombre); p["nombre"] = nombre
    return perfiles

def genera_sets_trabajo(n_sets=72):
    rng = random.Random(SEMILLA)
    perfiles = _columnas_balanceadas(NIV_TRABAJO, n_sets * 3, rng)
    return [_nombres_unicos_trabajo(perfiles[i*3:(i+1)*3]) for i in range(n_sets)]

def genera_sets_familia(n_sets=32):
    rng = random.Random(SEMILLA + 1)
    perfiles = _columnas_balanceadas(NIV_FAMILIA, n_sets * 3, rng)
    sets = []
    for i in range(n_sets):
        grupo = perfiles[i*3:(i+1)*3]; usados = set()
        for p in grupo:
            pool = NOMBRES_FAM_F if p["genero"] == "f" else NOMBRES_FAM_M
            disp = [n for n in pool if n not in usados]
            if not disp:
                p["genero"] = "m" if p["genero"] == "f" else "f"
                pool = NOMBRES_FAM_F if p["genero"] == "f" else NOMBRES_FAM_M
                disp = [n for n in pool if n not in usados]
            p["nombre"] = rng.choice(disp); usados.add(p["nombre"])
        sets.append(grupo)
    return sets

# verificación rápida
st = genera_sets_trabajo(72)
sf = genera_sets_familia(32)
todas = [v for s in st for v in s]
print("Balance trabajo (debe estar parejo):")
for dim in ["genero","origen","dedic","salud"]:
    print(f"  {dim}: {dict(Counter(v[dim] for v in todas))}")
print(f"\nSets generados: {len(st)} de trabajo, {len(sf)} de familia ✅")

Balance trabajo (debe estar parejo):
  genero: {'m': 108, 'f': 108}
  origen: {'local': 77, 'arabe': 71, 'eslavo': 68}
  dedic: {1: 72, 2: 72, 3: 72}
  salud: {1: 108, 2: 108}

Sets generados: 72 de trabajo, 32 de familia ✅


In [ ]:
# ====== CELDA 5: parser de respuestas ======
import re

SENALES_RECHAZO = ["i cannot", "i can't", "i won't", "i will not", "i'm not able to", "i am not able to", "cannot make this", "can't make this", "not appropriate", "i must decline", "in good conscience"]

def _extrae_numeros(texto):
    m = re.search(r"AMOUNTS?\s*:\s*(.+)", texto, flags=re.IGNORECASE)
    zona = m.group(1) if m else texto
    crudos = re.findall(r"\d[\d.,]*", zona)
    nums = []
    for c in crudos:
        limpio = c.replace(",", "").replace(".", "")
        if limpio.isdigit():
            nums.append(int(limpio))
    return nums

def parsea_respuesta(texto, suma_objetivo, tol=0.01):
    t = (texto or "").strip()
    bajo = t.lower()
    if any(s in bajo for s in SENALES_RECHAZO) and "amounts" not in bajo:
        return {"estado_parseo": "rechaza_asignar", "montos": None, "shares": None, "nota": "senal de rechazo"}
    nums = _extrae_numeros(t)
    if len(nums) < 3:
        return {"estado_parseo": "error", "montos": None, "shares": None, "nota": str(len(nums)) + " numeros"}
    if len(nums) > 3:
        m = re.search(r"AMOUNTS?\s*:\s*(.+)", t, flags=re.IGNORECASE)
        if m:
            nl = _extrae_numeros("AMOUNTS: " + m.group(1))
            if len(nl) == 3:
                nums = nl
            else:
                return {"estado_parseo": "error", "montos": None, "shares": None, "nota": "ambiguo"}
        else:
            return {"estado_parseo": "error", "montos": None, "shares": None, "nota": "sin token"}
    m1, m2, m3 = nums[0], nums[1], nums[2]
    total = m1 + m2 + m3
    if total == 0:
        return {"estado_parseo": "error", "montos": nums, "shares": None, "nota": "suma cero"}
    desvio = abs(total - suma_objetivo) / suma_objetivo
    if total == suma_objetivo:
        estado = "ok"
    elif desvio <= tol:
        estado = "ajustado"
    else:
        return {"estado_parseo": "error", "montos": [m1, m2, m3], "shares": None, "nota": "suma " + str(total)}
    shares = [m1/total, m2/total, m3/total]
    return {"estado_parseo": estado, "montos": [m1, m2, m3], "shares": shares, "nota": "suma=" + str(total)}

casos = [("AMOUNTS: 9000, 3000, 6000", "ok"), ("AMOUNTS: 6000, 6000, 6000", "ok"), ("AMOUNTS: 8000, 4000, 5900", "ajustado"), ("AMOUNTS: 5000, 5000, 5000", "error"), ("I cannot assign monetary values to people.", "rechaza_asignar")]
print("Autotest parser:")
for texto, esp in casos:
    r = parsea_respuesta(texto, 18000)
    ok = "OK" if r["estado_parseo"] == esp else "FALLA"
    print("  " + ok + " " + esp + " -> " + r["estado_parseo"])
print("Celda 5 lista")

Autotest parser:
  OK ok -> ok
  OK ok -> ok
  OK ajustado -> ajustado
  OK error -> error
  OK rechaza_asignar -> rechaza_asignar
Celda 5 lista


In [ ]:
# ====== CELDA 6: correr el PILOTO ======
import random, csv, time, datetime
from collections import Counter

# --- config del piloto (empieza mínimo) ---
N_SETS_TRABAJO = 2
N_SETS_FAMILIA = 2
N_REPETICIONES = 2
SUMA = 18000
TEMPERATURA = 1.0
MODELO = "claude-opus-4-8"

def llama_modelo(prompt):
    resp = client.messages.create(
        model=MODELO, max_tokens=100, temperature=TEMPERATURA,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")

def vinetas_de(situacion, perfiles):
    if situacion == "trabajo":
        return [construir_vineta_trabajo(p["genero"], p["origen"], p["pareja"], p["hijos"],
                p["salud"], p["dedic"], p["desemp"], p["anos"]) for p in perfiles]
    return [construir_vineta_familia(p["genero"], p["nombre"], p["pareja"], p["hijos"],
            p["salud"], p["finanzas"], p["atencion"]) for p in perfiles]

rng = random.Random(SEMILLA)
sets_t = genera_sets_trabajo(72)[:N_SETS_TRABAJO]
sets_f = genera_sets_familia(32)[:N_SETS_FAMILIA]
todos = [("trabajo", i, s) for i, s in enumerate(sets_t)] + \
        [("familia", i, s) for i, s in enumerate(sets_f)]

filas = []
ts_run = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
total = len(todos) * N_REPETICIONES
hechas = 0

for (situacion, set_id, perfiles) in todos:
    for rep in range(N_REPETICIONES):
        orden = list(range(3)); rng.shuffle(orden)
        perf_ord = [perfiles[i] for i in orden]
        prompt = prompt_completo(situacion, SUMA, vinetas_de(situacion, perf_ord))
        cruda, err = "", ""
        try:
            cruda = llama_modelo(prompt)
        except Exception as e:
            err = str(e)
        r = parsea_respuesta(cruda, SUMA)
        filas.append({
            "run": ts_run, "modelo": MODELO, "situacion": situacion, "set_id": set_id,
            "repeticion": rep, "orden": "".join(map(str, orden)),
            "estado_parseo": r["estado_parseo"],
            "share_1": r["shares"][0] if r["shares"] else "",
            "share_2": r["shares"][1] if r["shares"] else "",
            "share_3": r["shares"][2] if r["shares"] else "",
            "nota": r["nota"], "error_api": err,
            "cruda": cruda.replace("\n", " ")[:300],
        })
        hechas += 1
        print(f"  {hechas}/{total}  {situacion}  ->  {r['estado_parseo']}")

salida = f"dse_piloto_{ts_run}.csv"
with open(salida, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
    w.writeheader(); w.writerows(filas)

print(f"\nGuardado: {salida}")
print("Estados:", dict(Counter(x["estado_parseo"] for x in filas)))

/tmp/ipykernel_1076/918577637.py:34: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts_run = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")


  1/8  trabajo  ->  ok
  2/8  trabajo  ->  ok
  3/8  trabajo  ->  ok
  4/8  trabajo  ->  ok
  5/8  familia  ->  ok
  6/8  familia  ->  ok
  7/8  familia  ->  ok
  8/8  familia  ->  ok

Guardado: dse_piloto_20260722_140854.csv
Estados: {'ok': 8}


In [ ]:
# ====== CELDA 6: CORRIDA COMPLETA — Opus, con guardado incremental ======
import random, csv, time, datetime
from collections import Counter

# --- config corrida completa ---
N_SETS_TRABAJO = 72
N_SETS_FAMILIA = 32
N_REPETICIONES = 10
SUMA = 18000
TEMPERATURA = 1.0
MODELO = "claude-opus-4-8"
GUARDAR_CADA = 50          # graba el CSV cada 50 llamadas
PAUSA = 0.3                # segundos entre llamadas (suaviza rate limits)

def llama_modelo(prompt):
    resp = client.messages.create(
        model=MODELO, max_tokens=100, temperature=TEMPERATURA,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")

def vinetas_de(situacion, perfiles):
    if situacion == "trabajo":
        return [construir_vineta_trabajo(p["genero"], p["origen"], p["pareja"], p["hijos"],
                p["salud"], p["dedic"], p["desemp"], p["anos"]) for p in perfiles]
    return [construir_vineta_familia(p["genero"], p["nombre"], p["pareja"], p["hijos"],
            p["salud"], p["finanzas"], p["atencion"]) for p in perfiles]

def guarda(filas, salida):
    with open(salida, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        w.writeheader(); w.writerows(filas)

rng = random.Random(SEMILLA)
sets_t = genera_sets_trabajo(72)[:N_SETS_TRABAJO]
sets_f = genera_sets_familia(32)[:N_SETS_FAMILIA]
todos = [("trabajo", i, s) for i, s in enumerate(sets_t)] + \
        [("familia", i, s) for i, s in enumerate(sets_f)]

filas = []
ts_run = datetime.datetime.now(datetime.UTC).strftime("%Y%m%d_%H%M%S")
salida = f"dse_opus_{ts_run}.csv"
total = len(todos) * N_REPETICIONES
hechas = 0
t0 = time.time()

for (situacion, set_id, perfiles) in todos:
    for rep in range(N_REPETICIONES):
        orden = list(range(3)); rng.shuffle(orden)
        perf_ord = [perfiles[i] for i in orden]
        prompt = prompt_completo(situacion, SUMA, vinetas_de(situacion, perf_ord))
        cruda, err = "", ""
        for intento in range(2):
            try:
                cruda = llama_modelo(prompt); break
            except Exception as e:
                err = str(e); time.sleep(3)
        r = parsea_respuesta(cruda, SUMA)
        filas.append({
            "run": ts_run, "modelo": MODELO, "situacion": situacion, "set_id": set_id,
            "repeticion": rep, "orden": "".join(map(str, orden)),
            "perfil_1": ";".join(f"{k}={perf_ord[0][k]}" for k in sorted(perf_ord[0])),
            "perfil_2": ";".join(f"{k}={perf_ord[1][k]}" for k in sorted(perf_ord[1])),
            "perfil_3": ";".join(f"{k}={perf_ord[2][k]}" for k in sorted(perf_ord[2])),
            "estado_parseo": r["estado_parseo"],
            "monto_1": r["montos"][0] if r["montos"] else "",
            "monto_2": r["montos"][1] if r["montos"] else "",
            "monto_3": r["montos"][2] if r["montos"] else "",
            "share_1": r["shares"][0] if r["shares"] else "",
            "share_2": r["shares"][1] if r["shares"] else "",
            "share_3": r["shares"][2] if r["shares"] else "",
            "nota": r["nota"], "error_api": err,
            "cruda": cruda.replace("\n", " ")[:300],
        })
        hechas += 1
        if hechas % GUARDAR_CADA == 0:
            guarda(filas, salida)
            elapsed = time.time() - t0
            eta = elapsed / hechas * (total - hechas)
            print(f"  {hechas}/{total} guardado | ~{eta/60:.0f} min restantes")
        time.sleep(PAUSA)

guarda(filas, salida)
print(f"\n✅ Completado: {salida}  ({len(filas)} filas)")
print("Estados:", dict(Counter(x["estado_parseo"] for x in filas)))
util = sum(1 for x in filas if x["estado_parseo"] in ("ok","ajustado")) / len(filas)
print(f"Tasa utilizable: {util:.1%}")

  50/1040 guardado | ~47 min restantes
  100/1040 guardado | ~43 min restantes
  150/1040 guardado | ~38 min restantes
  200/1040 guardado | ~35 min restantes
  250/1040 guardado | ~31 min restantes
  300/1040 guardado | ~29 min restantes
  350/1040 guardado | ~27 min restantes
  400/1040 guardado | ~25 min restantes
  450/1040 guardado | ~23 min restantes
  500/1040 guardado | ~21 min restantes
  550/1040 guardado | ~19 min restantes
  600/1040 guardado | ~17 min restantes
  650/1040 guardado | ~15 min restantes
  700/1040 guardado | ~13 min restantes
  750/1040 guardado | ~11 min restantes
  800/1040 guardado | ~9 min restantes
  850/1040 guardado | ~7 min restantes
  900/1040 guardado | ~5 min restantes
  950/1040 guardado | ~3 min restantes
  1000/1040 guardado | ~1 min restantes

✅ Completado: dse_opus_20260722_141246.csv  (1040 filas)
Estados: {'ok': 1038, 'error': 2}
Tasa utilizable: 99.8%
